<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/CNN%2BLSTM%2BLOSO_Fine%20Tune%20murtaza%20v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
"""
WESAD SUBJECT-INDEPENDENT STRESS DETECTION FRAMEWORK (v4)
==========================================================
Leakage-safe per-fold hyperparameter fine-tuning
Multi-rate Multi-modal CNN-BiGRU + handcrafted features
Google Colab + Google Drive

IMPORTANT TERMINOLOGY
---------------------
This project uses "fine-tuning" to mean hyperparameter/training-protocol
fine-tuning of the existing CNN-BiGRU model. It is NOT external transfer
learning from ImageNet or another unrelated pretrained model.

MAIN PROTOCOL
-------------
Outer LOSO:
    one subject = test (completely untouched)

Inside each outer fold:
    1. Choose one rotating validation subject from the remaining subjects.
    2. Fit preprocessing scalers ONLY on the remaining training subjects.
    3. Search a small, curated set of hyperparameter configurations.
    4. Select the configuration by validation Macro F1.
    5. Take the best validation epoch.
    6. Refit a fresh model on ALL non-test subjects for that many epochs.
    7. Evaluate ONCE on the held-out test subject.

By default subject-baseline calibration is OFF because calculating test-subject
statistics before evaluation is a transductive operation. It can be enabled
explicitly, but should then be reported separately.

Expected WESAD structure:
    /content/drive/MyDrive/WESAD/WESAD.zip
or:
    /content/drive/MyDrive/WESAD.zip
"""

import os
import re
import json
import copy
import time
import random
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import signal
from scipy.stats import mode

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset
WESAD_ZIP_CANDIDATES = [
    "/content/drive/MyDrive/WESAD/WESAD.zip",
    "/content/drive/MyDrive/WESAD.zip",
]
WESAD_DIR = "/content/WESAD"

# Results
if os.path.exists("/content/drive/MyDrive"):
    OUTPUT_DIR = "/content/drive/MyDrive/WESAD_finetuning_results"
else:
    OUTPUT_DIR = "./WESAD_finetuning_results"

# Subjects
ALL_SUBJECTS = [f"S{i}" for i in range(2, 18) if i != 12]
VALID_LABELS_ORIG = [1, 2, 3]
LABEL_MAP = {1: 0, 2: 1, 3: 2}
CLASS_NAMES = ["Baseline", "Stress", "Amusement"]

# Sampling rates
BVP_SR = 64
ACC_SR = 32
EDA_SR = 4
TEMP_SR = 4
LABEL_SR = 700
HR_SR = 32
LR_SR = 4

# Windowing
WINDOW_SEC = 60
STEP_SEC = 5
HR_LEN = WINDOW_SEC * HR_SR
LR_LEN = WINDOW_SEC * LR_SR

# Preprocessing
USE_SUBJECT_BASELINE_CALIBRATION = False
MIN_HR_BPM = 30
MAX_HR_BPM = 220

# Training defaults / baseline candidate
BASELINE_CONFIG = {
    "lr": 7e-4,
    "weight_decay": 1e-4,
    "dropout": 0.35,
    "batch_size": 64,
    "aug_noise": 0.05,
    "aug_scale": 0.10,
}

# Curated search space. The first entry reproduces the original training setup.
TUNING_TRIALS = [
    BASELINE_CONFIG,
    {
        "lr": 1e-4,
        "weight_decay": 1e-4,
        "dropout": 0.45,
        "batch_size": 32,
        "aug_noise": 0.02,
        "aug_scale": 0.05,
    },
    {
        "lr": 3e-4,
        "weight_decay": 1e-4,
        "dropout": 0.35,
        "batch_size": 32,
        "aug_noise": 0.05,
        "aug_scale": 0.10,
    },
    {
        "lr": 3e-4,
        "weight_decay": 5e-4,
        "dropout": 0.45,
        "batch_size": 64,
        "aug_noise": 0.03,
        "aug_scale": 0.05,
    },
    {
        "lr": 5e-4,
        "weight_decay": 5e-4,
        "dropout": 0.25,
        "batch_size": 64,
        "aug_noise": 0.05,
        "aug_scale": 0.10,
    },
]

TUNING_EPOCHS = 15
TUNING_PATIENCE = 4
FINAL_MAX_EPOCHS = 40
FINAL_PATIENCE = 8

SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 3
GRAD_CLIP = 5.0

# Optional model saving
SAVE_MODELS = True

# =============================================================================
# REPRODUCIBILITY
# =============================================================================

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything()

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 80)
print("WESAD v4 — LEAKAGE-SAFE FINE-TUNING + LOSO")
print("=" * 80)
print("Device:", DEVICE)
print("Output:", OUTPUT_DIR)
print("Subjects:", ALL_SUBJECTS)
print("Test-subject baseline calibration:", USE_SUBJECT_BASELINE_CALIBRATION)
print()

# =============================================================================
# DATA DISCOVERY / LOADING
# =============================================================================

def is_colab():
    return "google.colab" in str(__import__("sys").modules)


def maybe_mount_google_drive():
    """Mount Google Drive automatically when running inside Colab."""
    if not is_colab():
        return False

    drive_root = "/content/drive/MyDrive"
    if os.path.exists(drive_root):
        return True

    try:
        from google.colab import drive
        print("Google Colab detected. Mounting Google Drive...")
        drive.mount("/content/drive")
        return os.path.exists(drive_root)
    except Exception as e:
        print("Could not auto-mount Google Drive:", repr(e))
        print("If your dataset is in Drive, run:")
        print("    from google.colab import drive")
        print("    drive.mount('/content/drive')")
        return False


def candidate_search_roots():
    """Return existing locations in which WESAD may have been uploaded/extracted."""
    roots = []

    # Current notebook/script location and working directory.
    roots.extend([
        os.getcwd(),
        "/mnt/data",
        "/content",
    ])

    # Google Drive locations, if mounted.
    if os.path.exists("/content/drive/MyDrive"):
        roots.extend([
            "/content/drive/MyDrive",
            "/content/drive/MyDrive/WESAD",
        ])

    # Configured extraction directory.
    roots.append(WESAD_DIR)

    # Remove duplicates and non-existing roots.
    result = []
    seen = set()
    for root in roots:
        root = os.path.abspath(root)
        if root not in seen and os.path.exists(root):
            seen.add(root)
            result.append(root)
    return result


def find_subject_pickle(subject_id, root=None):
    """Find a subject pickle recursively in common dataset locations."""
    roots = [root] if root else candidate_search_roots()
    candidates = []

    for search_root in roots:
        if not os.path.exists(search_root):
            continue
        try:
            for p in Path(search_root).rglob(f"{subject_id}.pkl"):
                # Avoid accidentally searching Python environments/cache folders.
                p_str = str(p)
                if any(part in p.parts for part in [
                    "site-packages", ".cache", ".git", "__pycache__"
                ]):
                    continue
                candidates.append(p_str)
        except (PermissionError, OSError):
            continue

    candidates = sorted(set(candidates), key=lambda x: (
        "WESAD" not in x.upper(),
        "MyDrive" not in x,
        len(x)
    ))

    if not candidates:
        raise FileNotFoundError(
            f"Could not find {subject_id}.pkl. Searched roots:\n"
            + "\n".join(f"  - {r}" for r in roots)
        )

    return candidates[0]


def find_wesad_zip():
    """Find WESAD.zip in Drive, /content, current directory, or /mnt/data."""
    candidates = list(WESAD_ZIP_CANDIDATES)

    # Add common local/notebook locations.
    candidates.extend([
        os.path.join(os.getcwd(), "WESAD.zip"),
        "/mnt/data/WESAD.zip",
        "/content/WESAD.zip",
    ])

    # Also search one level recursively in common roots.
    for root in candidate_search_roots():
        try:
            for p in Path(root).glob("WESAD.zip"):
                candidates.append(str(p))
        except (PermissionError, OSError):
            pass

    for p in candidates:
        if os.path.isfile(p):
            return p
    return None


def ensure_wesad_available():
    """Make sure at least S2.pkl can be found, extracting WESAD.zip if necessary."""
    maybe_mount_google_drive()

    try:
        path = find_subject_pickle("S2")
        print("WESAD subject files found.")
        print("Example S2 path:", path)
        return
    except FileNotFoundError:
        pass

    zip_path = find_wesad_zip()

    if zip_path is None:
        print("\nWESAD DATASET NOT FOUND.")
        print("\nThe script checked these locations:")
        for root in candidate_search_roots():
            print("  -", root)
        print("\nExpected ZIP names:")
        for p in WESAD_ZIP_CANDIDATES:
            print("  -", p)
        print("\nIf you are using Google Colab, run:")
        print("    from google.colab import drive")
        print("    drive.mount('/content/drive')")
        print("\nThen make sure WESAD.zip is at:")
        print("    /content/drive/MyDrive/WESAD/WESAD.zip")
        print("\nOr upload/extract the WESAD folder so that files such as")
        print("    S2.pkl")
        print("can be found.")
        raise FileNotFoundError(
            "WESAD dataset not found. See the messages above for the supported paths."
        )

    os.makedirs(WESAD_DIR, exist_ok=True)
    print("WESAD ZIP found:", zip_path)
    print("Extracting to:", WESAD_DIR)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(WESAD_DIR)

    try:
        path = find_subject_pickle("S2")
        print("Extraction successful.")
        print("Example S2 path:", path)
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "WESAD.zip was extracted, but S2.pkl could not be located. "
            "Check the ZIP's internal folder structure."
        ) from e


def load_subject(subject_id):
    import pickle
    path = find_subject_pickle(subject_id)
    with open(path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    return data


# =============================================================================
# SIGNAL UTILITIES
# =============================================================================

def as_1d(x):
    return np.asarray(x, dtype=np.float32).reshape(-1)


def as_acc_magnitude(acc):
    acc = np.asarray(acc, dtype=np.float32)
    if acc.ndim == 2 and acc.shape[1] >= 3:
        return np.sqrt(np.sum(acc[:, :3] ** 2, axis=1))
    return as_1d(acc)


def resample_signal(x, src_sr, dst_sr):
    x = as_1d(x)
    if src_sr == dst_sr:
        return x.astype(np.float32, copy=False)
    n = int(round(len(x) * dst_sr / src_sr))
    if n <= 1:
        return np.asarray([x[0]], dtype=np.float32)
    y = signal.resample(x, n)
    return np.asarray(y, dtype=np.float32)


def butter_lowpass(x, cutoff, fs, order=2):
    x = as_1d(x)
    nyq = fs / 2.0
    cutoff = min(cutoff, nyq * 0.99)
    b, a = signal.butter(order, cutoff / nyq, btype="low")
    if len(x) < max(len(a), len(b)) * 3:
        return x.copy()
    return signal.filtfilt(b, a, x).astype(np.float32)


def decompose_eda(eda):
    eda = as_1d(eda)
    tonic = butter_lowpass(eda, cutoff=0.05, fs=EDA_SR, order=2)
    phasic = eda - tonic
    return tonic.astype(np.float32), phasic.astype(np.float32)


# =============================================================================
# HANDCRAFTED FEATURES
# =============================================================================

FEATURE_NAMES = [
    "mean_hr",
    "std_hr",
    "rmssd",
    "hr_range",
    "beat_count",
    "mean_scl",
    "scr_count",
    "mean_scr_amp",
    "phasic_std",
    "acc_mean",
    "acc_std",
    "acc_max",
    "temp_mean",
    "temp_slope",
]


def hrv_features_from_bvp(bvp):
    bvp = as_1d(bvp)
    if len(bvp) < 10:
        return np.zeros(5, dtype=np.float32)

    x = bvp - np.mean(bvp)
    distance = max(1, int(HR_SR * 60.0 / MAX_HR_BPM))
    prominence = max(1e-6, 0.15 * np.std(x))

    peaks, _ = signal.find_peaks(x, distance=distance, prominence=prominence)

    if len(peaks) < 2:
        return np.zeros(5, dtype=np.float32)

    rr = np.diff(peaks) / HR_SR
    hr = 60.0 / rr
    valid = (hr >= MIN_HR_BPM) & (hr <= MAX_HR_BPM)
    hr = hr[valid]

    if len(hr) == 0:
        return np.zeros(5, dtype=np.float32)

    if len(rr) > 1:
        rr_valid = rr[valid]
        if len(rr_valid) > 1:
            diff_rr = np.diff(rr_valid)
            rmssd = np.sqrt(np.mean(diff_rr ** 2)) * 1000.0
        else:
            rmssd = 0.0
    else:
        rmssd = 0.0

    return np.asarray([
        np.mean(hr),
        np.std(hr),
        rmssd,
        np.ptp(hr),
        len(hr),
    ], dtype=np.float32)


def eda_features(eda):
    eda = as_1d(eda)
    tonic, phasic = decompose_eda(eda)

    # SCR peaks: simple, consistent detector matching the original framework.
    threshold = max(1e-5, 0.05 * np.std(phasic))
    peaks, props = signal.find_peaks(
        phasic,
        height=threshold,
        distance=max(1, int(EDA_SR * 1.0)),
    )
    heights = props.get("peak_heights", np.asarray([], dtype=np.float32))

    return np.asarray([
        np.mean(tonic),
        len(peaks),
        np.mean(heights) if len(heights) else 0.0,
        np.std(phasic),
    ], dtype=np.float32)


def acc_features(acc):
    acc = as_1d(acc)
    return np.asarray([
        np.mean(acc),
        np.std(acc),
        np.max(acc) if len(acc) else 0.0,
    ], dtype=np.float32)


def temp_features(temp):
    temp = as_1d(temp)
    if len(temp) < 2:
        return np.asarray([np.mean(temp), 0.0], dtype=np.float32)
    t = np.arange(len(temp), dtype=np.float32) / TEMP_SR
    slope = np.polyfit(t, temp, 1)[0]
    return np.asarray([np.mean(temp), slope], dtype=np.float32)


# =============================================================================
# WINDOWING
# =============================================================================

def majority_label(labels_window):
    labels_window = np.asarray(labels_window).reshape(-1)
    labels_window = labels_window[np.isin(labels_window, VALID_LABELS_ORIG)]
    if len(labels_window) == 0:
        return None
    m = mode(labels_window, keepdims=False).mode
    return int(m)


def window_subject_data(subject_id, data):
    wrist = data["signal"]["wrist"]
    label = as_1d(data["label"]).astype(np.int64)

    bvp = as_1d(wrist["BVP"])
    eda = as_1d(wrist["EDA"])
    temp = as_1d(wrist["TEMP"])
    acc = as_acc_magnitude(wrist["ACC"])

    # Convert all channels to the target multi-rate representations.
    bvp32 = resample_signal(bvp, BVP_SR, HR_SR)
    acc32 = resample_signal(acc, ACC_SR, HR_SR)
    eda4 = resample_signal(eda, EDA_SR, LR_SR)
    temp4 = resample_signal(temp, TEMP_SR, LR_SR)

    tonic4, phasic4 = decompose_eda(eda4)

    # Labels are resampled to 4 Hz only for window indexing.
    label4 = resample_signal(label.astype(np.float32), LABEL_SR, LR_SR)
    label4 = np.rint(label4).astype(np.int64)

    n_windows = min(
        (len(bvp32) - HR_LEN) // (STEP_SEC * HR_SR) + 1,
        (len(eda4) - LR_LEN) // (STEP_SEC * LR_SR) + 1,
        (len(temp4) - LR_LEN) // (STEP_SEC * LR_SR) + 1,
        (len(label4) - LR_LEN) // (STEP_SEC * LR_SR) + 1,
    )

    if n_windows <= 0:
        raise ValueError(f"{subject_id}: not enough signal length for 60s windows.")

    hr_windows, lr_windows, feat_windows, labels = [], [], [], []

    hr_step = STEP_SEC * HR_SR
    lr_step = STEP_SEC * LR_SR

    for i in range(n_windows):
        hs = i * hr_step
        ls = i * lr_step

        bvp_w = bvp32[hs:hs + HR_LEN]
        acc_w = acc32[hs:hs + HR_LEN]
        eda_t_w = tonic4[ls:ls + LR_LEN]
        eda_p_w = phasic4[ls:ls + LR_LEN]
        temp_w = temp4[ls:ls + LR_LEN]
        lab_w = label4[ls:ls + LR_LEN]

        orig_label = majority_label(lab_w)
        if orig_label is None:
            continue

        hr = np.stack([bvp_w, acc_w], axis=0).astype(np.float32)
        lr = np.stack([eda_t_w, eda_p_w, temp_w], axis=0).astype(np.float32)

        # HRV needs BVP at a rate consistent with the HR branch.
        feats = np.concatenate([
            hrv_features_from_bvp(bvp_w),
            eda_features(eda_t_w + eda_p_w),
            acc_features(acc_w),
            temp_features(temp_w),
        ]).astype(np.float32)

        hr_windows.append(hr)
        lr_windows.append(lr)
        feat_windows.append(feats)
        labels.append(LABEL_MAP[orig_label])

    if not labels:
        raise ValueError(f"{subject_id}: no valid windows found.")

    return {
        "hr": np.stack(hr_windows).astype(np.float32),
        "lr": np.stack(lr_windows).astype(np.float32),
        "features": np.stack(feat_windows).astype(np.float32),
        "labels": np.asarray(labels, dtype=np.int64),
    }


# =============================================================================
# OPTIONAL SUBJECT BASELINE CALIBRATION
# =============================================================================

def baseline_calibrate_3d(x, labels):
    x = np.asarray(x, dtype=np.float32)
    baseline = x[labels == 0]
    if len(baseline) == 0:
        return x

    flat = baseline.transpose(0, 2, 1).reshape(-1, x.shape[1])
    mu = flat.mean(axis=0)
    sd = flat.std(axis=0)
    sd[sd < 1e-6] = 1.0
    return ((x - mu[None, :, None]) / sd[None, :, None]).astype(np.float32)


def baseline_calibrate_2d(x, labels):
    x = np.asarray(x, dtype=np.float32)
    baseline = x[labels == 0]
    if len(baseline) == 0:
        return x
    mu = baseline.mean(axis=0)
    sd = baseline.std(axis=0)
    sd[sd < 1e-6] = 1.0
    return ((x - mu) / sd).astype(np.float32)


# =============================================================================
# LOAD ALL SUBJECTS
# =============================================================================

def prepare_all_subjects():
    ensure_wesad_available()
    subject_data = {}
    for sid in ALL_SUBJECTS:
        print(f"Preparing {sid} ...")
        raw = load_subject(sid)
        ds = window_subject_data(sid, raw)

        # Disabled by default for strict subject-independent evaluation.
        if USE_SUBJECT_BASELINE_CALIBRATION:
            ds["hr"] = baseline_calibrate_3d(ds["hr"], ds["labels"])
            ds["lr"] = baseline_calibrate_3d(ds["lr"], ds["labels"])
            ds["features"] = baseline_calibrate_2d(ds["features"], ds["labels"])

        subject_data[sid] = ds
        print(
            f"  windows={len(ds['labels'])}, "
            f"class_counts={np.bincount(ds['labels'], minlength=3).tolist()}"
        )
    return subject_data


# =============================================================================
# TRAIN-ONLY SCALING
# =============================================================================

class FoldScalers:
    def __init__(self):
        self.hr = StandardScaler()
        self.lr = StandardScaler()
        self.features = StandardScaler()

    @staticmethod
    def _fit_3d(scaler, arr):
        n, c, t = arr.shape
        scaler.fit(arr.transpose(0, 2, 1).reshape(-1, c))
        return scaler

    @staticmethod
    def _transform_3d(scaler, arr):
        n, c, t = arr.shape
        z = scaler.transform(arr.transpose(0, 2, 1).reshape(-1, c))
        return z.reshape(n, t, c).transpose(0, 2, 1).astype(np.float32)

    def fit(self, subjects):
        self._fit_3d(self.hr, np.concatenate([subjects[s]["hr"] for s in subjects]))
        self._fit_3d(self.lr, np.concatenate([subjects[s]["lr"] for s in subjects]))
        self.features.fit(np.concatenate([subjects[s]["features"] for s in subjects]))
        return self

    def transform_subject(self, ds):
        return {
            "hr": self._transform_3d(self.hr, ds["hr"]),
            "lr": self._transform_3d(self.lr, ds["lr"]),
            "features": self.features.transform(ds["features"]).astype(np.float32),
            "labels": ds["labels"].copy(),
        }


def merge_subjects(subjects, ids):
    return {
        "hr": np.concatenate([subjects[s]["hr"] for s in ids], axis=0),
        "lr": np.concatenate([subjects[s]["lr"] for s in ids], axis=0),
        "features": np.concatenate([subjects[s]["features"] for s in ids], axis=0),
        "labels": np.concatenate([subjects[s]["labels"] for s in ids], axis=0),
    }


def fit_transform_fold(subjects, fit_ids, transform_ids):
    scalers = FoldScalers().fit(subjects={s: subjects[s] for s in fit_ids})
    out = {}
    for sid in transform_ids:
        out[sid] = scalers.transform_subject(subjects[sid])
    return scalers, out


# =============================================================================
# DATASET / AUGMENTATION
# =============================================================================

class WESADDataset(Dataset):
    def __init__(self, data, augment=False, aug_noise=0.05, aug_scale=0.10):
        self.hr = torch.from_numpy(data["hr"])
        self.lr = torch.from_numpy(data["lr"])
        self.features = torch.from_numpy(data["features"])
        self.labels = torch.from_numpy(data["labels"]).long()
        self.augment = augment
        self.aug_noise = float(aug_noise)
        self.aug_scale = float(aug_scale)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        hr = self.hr[idx].clone()
        lr = self.lr[idx].clone()
        feat = self.features[idx]
        y = self.labels[idx]

        if self.augment:
            if np.random.rand() < 0.5:
                hr = hr + torch.randn_like(hr) * self.aug_noise
            if np.random.rand() < 0.5:
                lr = lr + torch.randn_like(lr) * self.aug_noise
            if np.random.rand() < 0.30:
                scale = 1.0 + np.random.uniform(-self.aug_scale, self.aug_scale)
                hr = hr * float(scale)

        return hr, lr, feat, y


def safe_drop_last(n_items, batch_size):
    # BatchNorm can fail with a final batch of size 1.
    return n_items > batch_size and (n_items % batch_size == 1)


def make_loader(data, config, train=False):
    ds = WESADDataset(
        data,
        augment=train,
        aug_noise=config["aug_noise"],
        aug_scale=config["aug_scale"],
    )
    return DataLoader(
        ds,
        batch_size=int(config["batch_size"]),
        shuffle=train,
        drop_last=safe_drop_last(len(ds), int(config["batch_size"])),
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


# =============================================================================
# MODEL
# =============================================================================

class HRBranch(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.pool = nn.AdaptiveAvgPool1d(60)
        self.gru = nn.GRU(
            input_size=64,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x)
        x = x.transpose(1, 2)
        x, _ = self.gru(x)
        return x.mean(dim=1)


class LRBranch(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = self.cnn(x)
        return torch.cat([
            x.mean(dim=2),
            x.max(dim=2).values,
        ], dim=1)


class FeatureBranch(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(len(FEATURE_NAMES), 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class MultiModalCNNBiGRU(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()
        self.hr_branch = HRBranch(dropout)
        self.lr_branch = LRBranch(dropout)
        self.feature_branch = FeatureBranch(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(128 + 128 + 32, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 3),
        )

    def forward(self, hr, lr, features):
        h = self.hr_branch(hr)
        l = self.lr_branch(lr)
        f = self.feature_branch(features)
        z = torch.cat([h, l, f], dim=1)
        return self.classifier(z)


# =============================================================================
# LOSS / METRICS
# =============================================================================

def make_class_weights(labels):
    counts = np.bincount(labels, minlength=3).astype(np.float32)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (len(counts) * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)


def evaluate(model, loader, criterion):
    model.eval()
    losses, ys, preds = [], [], []

    with torch.no_grad():
        for hr, lr, feat, y in loader:
            hr = hr.to(DEVICE, non_blocking=True)
            lr = lr.to(DEVICE, non_blocking=True)
            feat = feat.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(hr, lr, feat)
            loss = criterion(logits, y)

            losses.append(float(loss.item()))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            ys.extend(y.cpu().numpy())

    if not ys:
        return {
            "loss": float("inf"),
            "accuracy": 0.0,
            "balanced_accuracy": 0.0,
            "macro_f1": 0.0,
            "y_true": np.asarray([], dtype=np.int64),
            "y_pred": np.asarray([], dtype=np.int64),
        }

    y_true = np.asarray(ys)
    y_pred = np.asarray(preds)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


# =============================================================================
# TRAINING
# =============================================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    losses = []

    for hr, lr, feat, y in loader:
        hr = hr.to(DEVICE, non_blocking=True)
        lr = lr.to(DEVICE, non_blocking=True)
        feat = feat.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(hr, lr, feat)
        loss = criterion(logits, y)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        losses.append(float(loss.item()))

    return float(np.mean(losses)) if losses else float("inf")


def fit_with_validation(train_data, val_data, config, max_epochs, patience, seed):
    seed_everything(seed)

    train_loader = make_loader(train_data, config, train=True)
    val_loader = make_loader(val_data, config, train=False)

    if len(train_loader) == 0:
        raise ValueError(
            "Training DataLoader is empty. Reduce batch_size or check training data."
        )

    model = MultiModalCNNBiGRU(dropout=config["dropout"]).to(DEVICE)
    weights = make_class_weights(train_data["labels"])
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
    )

    best_state = None
    best_epoch = 1
    best_key = (-np.inf, -np.inf, np.inf)
    history = []
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val = evaluate(model, val_loader, criterion)

        key = (val["macro_f1"], val["balanced_accuracy"], -val["loss"])
        if key > best_key:
            best_key = key
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        scheduler.step(val["macro_f1"])

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val["loss"],
            "val_accuracy": val["accuracy"],
            "val_balanced_accuracy": val["balanced_accuracy"],
            "val_macro_f1": val["macro_f1"],
            "lr": optimizer.param_groups[0]["lr"],
        })

        print(
            f"    epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"val_f1={val['macro_f1']:.4f} | "
            f"val_bal={val['balanced_accuracy']:.4f}"
        )

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    best_val = evaluate(model, val_loader, criterion)

    return model, best_epoch, best_val, pd.DataFrame(history)


def fit_fixed_epochs(train_data, config, epochs, seed):
    seed_everything(seed)

    train_loader = make_loader(train_data, config, train=True)
    if len(train_loader) == 0:
        raise ValueError(
            "Training DataLoader is empty. Reduce batch_size or check training data."
        )

    model = MultiModalCNNBiGRU(dropout=config["dropout"]).to(DEVICE)
    weights = make_class_weights(train_data["labels"])
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
    )

    history = []
    for epoch in range(1, max(1, int(epochs)) + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        scheduler.step(train_loss)
        history.append({"epoch": epoch, "train_loss": train_loss})
        print(f"    final epoch {epoch:02d}/{epochs} | train_loss={train_loss:.4f}")

    return model, pd.DataFrame(history)


# =============================================================================
# FOLD PROTOCOL
# =============================================================================

def choose_rotating_validation_subject(test_subject, candidates):
    candidates = sorted(candidates)
    if not candidates:
        raise ValueError("No candidates available for validation subject.")
    num = int(re.sub(r"\D", "", test_subject))
    return candidates[num % len(candidates)]


def run_outer_fold(all_subjects_data, test_subject, fold_index):
    all_ids = sorted(all_subjects_data.keys())
    non_test = [s for s in all_ids if s != test_subject]
    val_subject = choose_rotating_validation_subject(test_subject, non_test)
    train_ids = [s for s in non_test if s != val_subject]

    print("\n" + "=" * 80)
    print(f"OUTER FOLD {fold_index} | TEST={test_subject} | VAL={val_subject}")
    print("TRAIN:", train_ids)
    print("=" * 80)

    # Strict preprocessing: scalers are fitted only on training subjects.
    _, scaled = fit_transform_fold(
        all_subjects_data,
        fit_ids=train_ids,
        transform_ids=train_ids + [val_subject],
    )
    train_data = merge_subjects(scaled, train_ids)
    val_data = scaled[val_subject]

    trial_rows = []
    best_trial = None

    for trial_idx, config in enumerate(TUNING_TRIALS, start=1):
        print(f"\n  [FINE-TUNE] Trial {trial_idx}/{len(TUNING_TRIALS)}")
        print("  Config:", config)

        trial_seed = SEED + fold_index * 1000 + trial_idx
        start = time.time()

        model, best_epoch, val, history = fit_with_validation(
            train_data=train_data,
            val_data=val_data,
            config=config,
            max_epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
            seed=trial_seed,
        )

        elapsed = time.time() - start
        row = {
            "fold": fold_index,
            "test_subject": test_subject,
            "val_subject": val_subject,
            "trial": trial_idx,
            **config,
            "best_epoch": best_epoch,
            "val_loss": val["loss"],
            "val_accuracy": val["accuracy"],
            "val_balanced_accuracy": val["balanced_accuracy"],
            "val_macro_f1": val["macro_f1"],
            "time_sec": elapsed,
        }
        trial_rows.append(row)

        print(
            f"  -> val_macro_f1={val['macro_f1']:.4f}, "
            f"val_bal={val['balanced_accuracy']:.4f}, "
            f"best_epoch={best_epoch}, time={elapsed/60:.1f} min"
        )

        # Selection rule: Macro F1, then balanced accuracy, then lower loss.
        score = (val["macro_f1"], val["balanced_accuracy"], -val["loss"])
        if best_trial is None or score > best_trial["score"]:
            best_trial = {
                "score": score,
                "config": copy.deepcopy(config),
                "best_epoch": best_epoch,
                "val": val,
            }

    trial_df = pd.DataFrame(trial_rows)
    trial_path = os.path.join(
        OUTPUT_DIR, f"tuning_trials_fold_{fold_index:02d}_{test_subject}.csv"
    )
    trial_df.to_csv(trial_path, index=False)

    print("\n  SELECTED FINE-TUNED CONFIG:")
    print(best_trial["config"])
    print("  Validation Macro F1:", best_trial["val"]["macro_f1"])
    print("  Validation best epoch:", best_trial["best_epoch"])

    # -------------------------------------------------------------------------
    # FINAL REFIT
    # -------------------------------------------------------------------------
    # IMPORTANT: validation subject is now allowed back into training.
    # Test subject remains untouched.
    final_train_ids = non_test

    _, final_scaled = fit_transform_fold(
        all_subjects_data,
        fit_ids=final_train_ids,
        transform_ids=final_train_ids + [test_subject],
    )

    final_train_data = merge_subjects(final_scaled, final_train_ids)
    final_test_data = final_scaled[test_subject]

    print("\n  FINAL REFIT on ALL NON-TEST SUBJECTS")
    final_model, final_history = fit_fixed_epochs(
        train_data=final_train_data,
        config=best_trial["config"],
        epochs=best_trial["best_epoch"],
        seed=SEED + 10000 + fold_index,
    )

    criterion = nn.CrossEntropyLoss(
        weight=make_class_weights(final_train_data["labels"])
    )
    test_loader = make_loader(
        final_test_data,
        best_trial["config"],
        train=False,
    )
    test_metrics = evaluate(final_model, test_loader, criterion)

    result = {
        "fold": fold_index,
        "test_subject": test_subject,
        "val_subject": val_subject,
        "test_accuracy": test_metrics["accuracy"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "selected_lr": best_trial["config"]["lr"],
        "selected_weight_decay": best_trial["config"]["weight_decay"],
        "selected_dropout": best_trial["config"]["dropout"],
        "selected_batch_size": best_trial["config"]["batch_size"],
        "selected_aug_noise": best_trial["config"]["aug_noise"],
        "selected_aug_scale": best_trial["config"]["aug_scale"],
        "selected_epoch": best_trial["best_epoch"],
    }

    # Save final model for reproducibility.
    if SAVE_MODELS:
        model_path = os.path.join(
            OUTPUT_DIR, f"final_model_fold_{fold_index:02d}_{test_subject}.pt"
        )
        torch.save(
            {
                "model_state_dict": final_model.state_dict(),
                "config": best_trial["config"],
                "best_epoch": best_trial["best_epoch"],
                "test_subject": test_subject,
                "val_subject": val_subject,
                "feature_names": FEATURE_NAMES,
                "class_names": CLASS_NAMES,
                "seed": SEED + 10000 + fold_index,
            },
            model_path,
        )

    # Save final training history.
    final_history.to_csv(
        os.path.join(
            OUTPUT_DIR, f"final_history_fold_{fold_index:02d}_{test_subject}.csv"
        ),
        index=False,
    )

    # Store predictions for global evaluation.
    result["_y_true"] = test_metrics["y_true"]
    result["_y_pred"] = test_metrics["y_pred"]

    print(
        f"\n  TEST {test_subject}: "
        f"Acc={test_metrics['accuracy']:.4f}, "
        f"BalAcc={test_metrics['balanced_accuracy']:.4f}, "
        f"MacroF1={test_metrics['macro_f1']:.4f}"
    )

    return result, trial_df


# =============================================================================
# REPORTING
# =============================================================================

def save_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_pct = cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1) * 100.0

    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        os.path.join(OUTPUT_DIR, "confusion_matrix_counts.csv")
    )
    pd.DataFrame(cm_pct, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        os.path.join(OUTPUT_DIR, "confusion_matrix_row_percent.csv")
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_pct)
    fig.colorbar(im, ax=ax, label="Row percentage")
    ax.set_xticks(range(3), CLASS_NAMES, rotation=25, ha="right")
    ax.set_yticks(range(3), CLASS_NAMES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("LOSO Confusion Matrix — Row Percent")

    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm_pct[i, j]:.1f}%", ha="center", va="center")

    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "confusion_matrix_row_percent.png"),
        dpi=180,
        bbox_inches="tight",
    )
    plt.close(fig)

    return cm, cm_pct


def run_all_loso(subjects_data):
    fold_results = []
    all_trials = []
    all_true = []
    all_pred = []

    for fold_idx, test_subject in enumerate(sorted(subjects_data.keys()), start=1):
        result, trials = run_outer_fold(
            subjects_data,
            test_subject=test_subject,
            fold_index=fold_idx,
        )

        fold_results.append(result)
        all_trials.append(trials)
        all_true.extend(result["_y_true"].tolist())
        all_pred.extend(result["_y_pred"].tolist())

        # Checkpoint after every fold.
        clean = [{k: v for k, v in r.items() if not k.startswith("_")} for r in fold_results]
        pd.DataFrame(clean).to_csv(
            os.path.join(OUTPUT_DIR, "fold_results_checkpoint.csv"),
            index=False,
        )

    fold_df = pd.DataFrame([
        {k: v for k, v in r.items() if not k.startswith("_")}
        for r in fold_results
    ])
    trials_df = pd.concat(all_trials, ignore_index=True)

    y_true = np.asarray(all_true, dtype=np.int64)
    y_pred = np.asarray(all_pred, dtype=np.int64)

    overall = {
        "overall_accuracy": float(accuracy_score(y_true, y_pred)),
        "overall_balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "overall_macro_f1": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "mean_subject_accuracy": float(fold_df["test_accuracy"].mean()),
        "std_subject_accuracy": float(fold_df["test_accuracy"].std(ddof=0)),
        "mean_subject_balanced_accuracy": float(
            fold_df["test_balanced_accuracy"].mean()
        ),
        "std_subject_balanced_accuracy": float(
            fold_df["test_balanced_accuracy"].std(ddof=0)
        ),
        "mean_subject_macro_f1": float(fold_df["test_macro_f1"].mean()),
        "std_subject_macro_f1": float(fold_df["test_macro_f1"].std(ddof=0)),
        "num_subjects": int(len(fold_df)),
        "test_subjects": sorted(subjects_data.keys()),
        "baseline_candidate": BASELINE_CONFIG,
        "num_tuning_trials_per_fold": len(TUNING_TRIALS),
        "tuning_selection_metric": "validation_macro_f1",
        "subject_baseline_calibration": USE_SUBJECT_BASELINE_CALIBRATION,
    }

    fold_df.to_csv(os.path.join(OUTPUT_DIR, "fold_results.csv"), index=False)
    trials_df.to_csv(os.path.join(OUTPUT_DIR, "all_tuning_trials.csv"), index=False)

    with open(os.path.join(OUTPUT_DIR, "overall_results.json"), "w", encoding="utf-8") as f:
        json.dump(overall, f, indent=2)

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )
    with open(
        os.path.join(OUTPUT_DIR, "classification_report.txt"),
        "w",
        encoding="utf-8",
    ) as f:
        f.write(report)

    save_confusion_matrix(y_true, y_pred)

    print("\n" + "=" * 80)
    print("FINAL LOSO RESULTS")
    print("=" * 80)
    print(f"Overall Accuracy       : {overall['overall_accuracy']:.4f}")
    print(f"Overall Balanced Acc. : {overall['overall_balanced_accuracy']:.4f}")
    print(f"Overall Macro F1      : {overall['overall_macro_f1']:.4f}")
    print(
        f"Mean Subject Accuracy : {overall['mean_subject_accuracy']:.4f} "
        f"+/- {overall['std_subject_accuracy']:.4f}"
    )
    print(
        f"Mean Subject Macro F1 : {overall['mean_subject_macro_f1']:.4f} "
        f"+/- {overall['std_subject_macro_f1']:.4f}"
    )
    print("\nClassification report:\n")
    print(report)
    print("\nSaved results to:", OUTPUT_DIR)

    return fold_df, trials_df, overall


# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    subjects_data = prepare_all_subjects()
    run_all_loso(subjects_data)


WESAD v4 — LEAKAGE-SAFE FINE-TUNING + LOSO
Device: cuda
Output: ./WESAD_finetuning_results
Subjects: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
Test-subject baseline calibration: False

Google Colab detected. Mounting Google Drive...
Mounted at /content/drive
WESAD ZIP found: /content/drive/MyDrive/WESAD/WESAD.zip
Extracting to: /content/WESAD
Extraction successful.
Example S2 path: /content/WESAD/WESAD/S2/S2.pkl
Preparing S2 ...
  windows=533, class_counts=[265, 159, 109]
Preparing S3 ...
  windows=512, class_counts=[285, 140, 87]
Preparing S4 ...
  windows=559, class_counts=[267, 172, 120]
Preparing S5 ...
  windows=492, class_counts=[264, 141, 87]
Preparing S6 ...
  windows=560, class_counts=[284, 166, 110]
Preparing S7 ...
  windows=487, class_counts=[261, 140, 86]
Preparing S8 ...
  windows=569, class_counts=[246, 216, 107]
Preparing S9 ...
  windows=581, class_counts=[249, 201, 131]
Preparing S10 ...
  windows=526, class_coun